## Random Forest Agent

This notebook mirrors `src/04_random_forest.py` so you can:

- **Train** a random forest on all `*.jsonl` files in a data folder (state → action).
- **Deduplicate** raw states and resolve **conflicting labels** with a majority vote.
- **Evaluate** with **session-level splits** (`GroupShuffleSplit`) so hold-out metrics reflect generalization to **new recordings**, not random frames from the same session.
- **Build** custom features via `src/processing/feature_engineering.py` (same logic as the training script).

**Recording your own sessions:** run `src/02_record_gameplay.py` (e.g. `python src/02_record_gameplay.py` from the repo root) to record gameplay as `.jsonl` files. You need those recordings to train this kind of imitation-learning agent; save or copy them under `DATA_DIR` (top-level `*.jsonl` only).

Observation layout is the same as elsewhere: see `00_data_exploration.ipynb`. Training only uses top-level `*.jsonl` in the chosen directory (not subfolders); point `DATA_DIR` at a folder that contains your recordings.

In [ ]:
import sys
from pathlib import Path
from collections import Counter, defaultdict

sys.path.insert(0, "..")
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

from processing.frame_visualizer import FrameVisualizer
from utils import load_observations_by_session, setup_environment

In [ ]:
def plot_state_duplicates(total_data, unique_count):
    import matplotlib.pyplot as plt

    duplicate_count = total_data - unique_count
    plt.figure(figsize=(6, 6))
    plt.pie(
        [unique_count, duplicate_count],
        labels=["Unique", "Duplicates"],
        autopct="%1.1f%%",
        startangle=90,
        explode=(0, 0.05),
    )
    plt.title(f"Dataset Deduplication (total={total_data})")
    plt.axis("equal")
    plt.show()

def plot_action_distribution(actions):
    plt.hist(actions, bins=np.arange(-0.5, max(actions) + 1.5, 1), rwidth=0.8)
    plt.xlabel("Action")
    plt.ylabel("Frequency")
    plt.title("Distribution of Actions in the Dataset")
    plt.xticks(np.arange(max(actions) + 1))
    plt.show()

In [ ]:
data_path = "../data/"
all_data = {}

for i, data_file in enumerate(Path(data_path).glob("*.jsonl")):
    observations_by_session = load_observations_by_session(data_file)
    for session_id, all_steps in observations_by_session.items():
        all_data[f"{i}_{session_id}"] = all_steps

We deduplicate **raw** observations (same state vector) and pick a **single** label with a majority vote so the model does not see contradictory (state → action) pairs.

That does **not** remove **session-level** correlation: frames from one recording are still similar. For a more honest hold-out score, we split train/validation by **session** (`GroupShuffleSplit`), not by shuffling individual frames.

In [ ]:
import hashlib

grouped = defaultdict(lambda: {"state": None, "actions": [], "group": None})

# Deduplicate by raw state hash; keep majority action. Track a session `group`
# (last session that wrote this state) for GroupShuffleSplit later.
total_samples = 0
for session_key, all_steps in all_data.items():
    for step in all_steps:
        state = step["state"]
        state_hash = hashlib.sha256(str(state).encode()).hexdigest()

        grouped[state_hash]["state"] = state
        grouped[state_hash]["actions"].append(step["action"])
        grouped[state_hash]["group"] = session_key
        total_samples += 1

aggregated_data = []
for item in grouped.values():
    action_counts = Counter(item["actions"])
    majority_action = action_counts.most_common(1)[0][0]

    aggregated_data.append({
        "state": item["state"],
        "action": majority_action,
        "group": item["group"],
    })

plot_state_duplicates(total_samples, len(aggregated_data))

In [ ]:
def load_training_xy(data):
    X, y, groups = [], [], []
    for row in data:
        X.append(row["state"])
        y.append(row["action"])
        groups.append(row["group"])
    return X, np.asarray(y, dtype=np.int64), np.asarray(groups)


X, y, groups = load_training_xy(aggregated_data)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train = np.asarray([X[i] for i in train_idx])
X_test = np.asarray([X[i] for i in test_idx])
y_train, y_test = y[train_idx], y[test_idx]


The issue we have right now is that there are more samples with action 0 (don't jump) than with 1 (jump)

In [ ]:
plot_action_distribution(y_train)

In this case, we'll use a **class-weight** map for training.

Misclassifying **action = 0** (no jump) is weighted at `1.0`. Misclassifying **action = 1** (jump) is weighted higher by a **ratio** so the model pays more attention to the rarer jump class.

**Important:** `ratio` is computed from **aggregated** labels (`y` after dedupe / majority), not from the raw per-frame action list, so it matches what the model actually fits.

We do this to **compensate** for class imbalance: during a playthrough the jump key is pressed less often than not.

In [ ]:
nb_ones = int(y_train.sum())
nb_zeros = int(len(y_train) - nb_ones)
ratio = 1.0 if nb_ones == 0 else nb_zeros / nb_ones
class_weights = {0: 1.0, 1: ratio}

class_weights

In [ ]:
model = RandomForestClassifier(
    n_estimators=10,
    random_state=42,
    max_depth=10,
    class_weight=class_weights
)

model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)

print(f"Random forest accuracy (hold-out, raw state, session split): {accuracy:.2f}")

In [ ]:
def agent_brain(model, step_count, observation):
    # You need to press jump on the first frame to start walking.
    if step_count == 0:
        return [1]

    pred = model.predict([observation])[0]
    return [int(pred)]


env = setup_environment()
obs = env.reset()
nb_agents = len(obs["obs"])

step_count = 0
while True:
    actions = [
        agent_brain(model, step_count, obs["obs"][i]) for i in range(nb_agents)
    ]
    actions = np.array(actions, dtype=np.int64)
    obs, reward, done, info = env.step(actions)
    if any(done):
        break
    step_count += 1

env.close()

Now we may also want to play around with some custom features

In [ ]:
def compute_can_jump(sensor_values, session_info):
    """
    The player can jump if:
      - After jumping, it released the jump button and touched the floor
      - After jumping, it released the jump button and has a powerup
    """
    on_floor = bool(sensor_values["on_floor"])
    has_powerup = bool(sensor_values["has_powerup"])
    jump_pressed = int(session_info["prev_action"]) == 1  # jump action id
    
    return float(has_powerup or (on_floor and not jump_pressed))

In [ ]:
def compute_wall_sensors(grid, wall_index):
    wall_distance_sensors = []
    
    right_wall_distance = 1
    for i, right_cells in enumerate([grid[3][3], grid[3][4], grid[3][5], grid[3][6]]):
        if right_cells == wall_index:
            right_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(right_wall_distance)

    left_wall_distance = 1
    for i, left_cells in enumerate([grid[3][3], grid[3][2], grid[3][1], grid[3][0]]):
        if left_cells == wall_index:
            left_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(left_wall_distance)
    
    up_wall_distance = 1
    for i, up_cells in enumerate([grid[3][3], grid[2][3], grid[1][3], grid[0][3]]):
        if up_cells == wall_index:
            up_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(up_wall_distance)

    down_wall_distance = 1
    for i, down_cells in enumerate([grid[3][3], grid[4][3], grid[5][3], grid[6][3]]):
        if down_cells == wall_index:
            down_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(down_wall_distance)

    diagonal_up_left_wall_distance = 1
    for i, up_left_cells in enumerate([grid[3][3], grid[2][2], grid[1][1], grid[0][0]]):
        if up_left_cells == wall_index:
            diagonal_up_left_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(diagonal_up_left_wall_distance)

    diagonal_up_right_wall_distance = 1
    for i, up_right_cells in enumerate([grid[3][3], grid[2][4], grid[1][5], grid[0][6]]):
        if up_right_cells == wall_index:
            diagonal_up_right_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(diagonal_up_right_wall_distance)

    diagonal_down_left_wall_distance = 1
    for i, down_left_cells in enumerate([grid[3][3], grid[4][2], grid[5][1], grid[6][0]]):
        if down_left_cells == wall_index:
            diagonal_down_left_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(diagonal_down_left_wall_distance)

    diagonal_down_right_wall_distance = 1
    for i, down_right_cells in enumerate([grid[3][3], grid[4][4], grid[5][5], grid[6][6]]):
        if down_right_cells == wall_index:
            diagonal_down_right_wall_distance = i * 0.25
            break
    wall_distance_sensors.append(diagonal_down_right_wall_distance)

    return wall_distance_sensors

In [ ]:
def engineer_features(observation, session_info=None):
    visualizer = FrameVisualizer()
    parsed_obs = visualizer.parse_observation(observation)
    grid = parsed_obs["grid"]
    sensor_values = parsed_obs["sensor_values"]
    sensors = list(sensor_values.values())

    # Create features for the player's ability to jump
    can_jump = compute_can_jump(sensor_values, session_info)

    # Create features for walls around the agent
    wall_index = visualizer.get_class_index("Wall")
    wall_distance_sensors = compute_wall_sensors(grid, wall_index)

    return [can_jump] + wall_distance_sensors + sensors

Update training data to also use the features we created

In [ ]:
def load_training_xy(data):
    X, y, groups = [], [], []
    session_info = {"prev_action": 0}
    for row in data:
        X.append(engineer_features(row["state"], session_info))
        y.append(row["action"])
        groups.append(row["group"])
        session_info["prev_action"] = row["action"]
    return X, np.asarray(y, dtype=np.int64), np.asarray(groups)


X, y, groups = load_training_xy(aggregated_data)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
X_train = np.asarray([X[i] for i in train_idx])
X_test = np.asarray([X[i] for i in test_idx])
y_train, y_test = y[train_idx], y[test_idx]

In [ ]:
nb_ones = int(y_train.sum())
nb_zeros = int(len(y_train) - nb_ones)
ratio = 1.0 if nb_ones == 0 else nb_zeros / nb_ones
class_weights = {0: 1.0, 1: ratio}

class_weights

In [ ]:
model = RandomForestClassifier(
    n_estimators=20,
    random_state=42,
    max_depth=10,
    class_weight=class_weights
)

model.fit(X_train, y_train)
accuracy = model.score(X_test, y_test)

print(f"Random forest accuracy (hold-out): {accuracy:.2f}")

In [ ]:
def agent_brain(model, step_count, observation, session_info, done=False):
    # You need to press jump on the first frame to start walking.
    if step_count == 0 or done:
        return [1]

    engineered_features = engineer_features(observation, session_info)
    pred = model.predict([engineered_features])[0]
    return [int(pred)]


env = setup_environment()
obs = env.reset()
nb_agents = len(obs["obs"])

step_count = 0
session_infos = [{"prev_action": 0} for i in range(nb_agents)]
done = [False] * nb_agents
while True:
    actions = [
        agent_brain(model, step_count, obs["obs"][i], session_infos[i], done[i]) for i in range(nb_agents)
    ]
    actions = np.array(actions, dtype=np.int64)
    obs, reward, done, info = env.step(actions)

    # Update the session info for each agent
    for i, action in enumerate(actions):
        session_infos[i]["prev_action"] = action[0]

    if any(done):
        break
    step_count += 1

env.close()